# C02 — Engenharia de Prompt e Saída Controlada

**Disciplina:** Sistemas Cognitivos com Large Language Models (INFNET, 26E2_3)
**Autor:** Anderson Felipe Paixão Corrêa
**Projeto:** O Direito como Dado — RAG hierarquia- e vigência-aware sobre o microssistema
penal federal brasileiro

Esta notebook cobre o **ponto 2 da rubrica** (engenharia de prompt + saída controlada). Ela
**empacota** módulos já implementados e testados do pacote `direito_dados`
(`direito_dados.generation`) — nenhuma lógica de prompt/parsing é reimplementada aqui,
apenas chamada e narrada.

O objetivo não é só "mostrar 3 técnicas soltas": é contar a **história real de iteração**
deste projeto até chegar ao prompt de produção usado por `answer_question`
(`c05_rag_pipeline.ipynb`). Cada etapa tem um **critério de qualidade explícito**: *o modelo
preenche corretamente o array `citations` e define `abstained` de forma coerente com o
contexto fornecido?*

1. **Técnica 1 — zero-shot**: instrução simples, sem exemplo, sem saída controlada.
2. **Técnica 2 — JSON não-guiado**: `format="json"` do Ollama garante JSON válido, mas sem
   contrato de chaves — mostra a falha de "schema errado".
3. **Técnica 3 — few-shot + saída controlada por schema**: `SYSTEM_PROMPT` traz um exemplo
   preenchido (few-shot) *e* a chamada usa `format=<json schema>` do Ollama, que restringe a
   saída à forma exata esperada por `parse_answer`.
4. `parse_answer` — parsing tolerante (```json fences, prosa ao redor) com **fail-safe para
   abstenção** em saída não-parseável.

## Setup

Importa apenas a API pública de `direito_dados.generation` (a mesma usada pelos testes do
projeto) e dois dispositivos reais do Código Penal (via `direito_dados.corpus`) para servir
de contexto nos prompts — sem precisar construir o índice vetorial completo, que é
demonstrado em `c03_embeddings_busca.ipynb` e `c05_rag_pipeline.ipynb`.

In [1]:
import sys
from pathlib import Path

# garante que o pacote local seja importável mesmo se o kernel iniciar em outro cwd
_repo_root = Path.cwd()
if not (_repo_root / "direito_dados").exists():
    _repo_root = Path(__file__).resolve().parent if "__file__" in globals() else _repo_root
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from direito_dados.corpus import load_corpus, NORMS
from direito_dados.generation.llm import OllamaClient, ollama_available, ollama_has_model
from direito_dados.generation.prompt import SYSTEM_PROMPT, build_user_prompt
from direito_dados.generation.parse import parse_answer
from direito_dados.retrieval.index import Result

MODEL = "llama3.1:8b"
assert ollama_available(), "Ollama precisa estar rodando em localhost:11434"
assert ollama_has_model(MODEL), f"Modelo {MODEL} precisa estar baixado (`ollama pull {MODEL}`)"
llm = OllamaClient(model=MODEL)
# OllamaClient tem json_format=True por padrão (todo o pipeline de produção espera JSON) —
# para a Técnica 1 (zero-shot "de verdade", prosa livre, sem nenhuma restrição de formato)
# precisamos de uma instância com json_format=False.
llm_prose = OllamaClient(model=MODEL, json_format=False)
print(f"Ollama disponível, modelo {MODEL} pronto.")

Ollama disponível, modelo llama3.1:8b pronto.


In [2]:
corpus = load_corpus("data/raw", specs=[NORMS["CP"]])
cp = corpus.norms[0]
art121 = next(a for a in cp.articles if a.number == "121")
art155 = next(a for a in cp.articles if a.number == "155")

results = [
    Result(id="CP:art121", text=art121.text, citation=art121.citation, score=1.0,
           metadata={"status": art121.status.value, "citation": art121.citation}),
    Result(id="CP:art155", text=art155.text, citation=art155.citation, score=1.0,
           metadata={"status": art155.status.value, "citation": art155.citation}),
]
QUESTION = "Qual a pena para quem mata alguém?"
print(f"Contexto: {[r.id for r in results]}")
print(f"Pergunta: {QUESTION!r}")

Contexto: ['CP:art121', 'CP:art155']
Pergunta: 'Qual a pena para quem mata alguém?'


## Técnica 1 — Zero-shot (instrução simples, sem exemplo, sem saída controlada)

A primeira iteração real deste projeto foi a mais ingênua possível: pedir a resposta em
linguagem natural, sem system prompt, sem exemplo e sem `format`. É o baseline contra o qual
as técnicas seguintes são comparadas.

**Critério de qualidade:** o resultado precisa ser JSON parseável com `citations`
preenchido corretamente. Como veremos, zero-shot falha nesse critério — não por a resposta
estar "errada" em conteúdo, mas por não ser estruturada.

In [3]:
zero_shot_prompt = (
    f"Contexto:\n{results[0].citation}: {results[0].text}\n\n"
    f"{results[1].citation}: {results[1].text}\n\n"
    f"Pergunta: {QUESTION}\nResponda com base apenas no contexto acima."
)
raw_zero_shot = llm_prose.generate(zero_shot_prompt)  # sem system, sem format -> prosa livre
print("--- Saída bruta (zero-shot) ---")
print(raw_zero_shot)

--- Saída bruta (zero-shot) ---
A pena prevista para o homicídio (matar alguém) não está explicitamente mencionada nas normas apresentadas, mas é possível inferir que a pena seria de reclusão por um período que varia de acordo com as circunstâncias do crime.


In [4]:
parsed_zero_shot = parse_answer(raw_zero_shot)
print(f"parse_ok:   {parsed_zero_shot.parse_ok}")
print(f"abstained:  {parsed_zero_shot.abstained}")
print(f"citations:  {parsed_zero_shot.citations}")
print(f"confidence: {parsed_zero_shot.confidence}")

parse_ok:   False
abstained:  True
citations:  []
confidence: 0.0


**Leitura:** o modelo respondeu corretamente *em conteúdo* (cita a pena de reclusão de 6 a
20 anos), mas em prosa livre. `parse_answer` não encontra um objeto JSON balanceado no
texto, então falha **seguro**: `parse_ok=False`, `abstained=True`, `citations=[]` — mesmo
o modelo "sabendo a resposta", o pipeline downstream não consegue extrair `answer`/
`citations` de forma confiável. Essa é a primeira lição de iteração: **texto livre não é
suficiente para um sistema que precisa citar fontes verificáveis**.

## Técnica 2 — JSON não-guiado (`format="json"`, sem contrato de chaves)

A segunda iteração adicionou `format="json"` (modo JSON do Ollama, que garante sintaxe
válida) mas usou um system prompt simples, **sem** fixar o contrato exato de chaves
(`answer`/`citations`/`abstained`/...) nem listar os ids citáveis. É o meio-termo real que
o projeto passou antes de chegar ao prompt final — sintaticamente JSON válido, mas com
schema divergente do que `parse_answer`/`answer_question` esperam.

**Critério de qualidade:** mesmo teste — `citations` precisa vir preenchido com os ids
exatos (`"CP:art121"`). Aqui a saída **é** JSON, mas o schema pode não bater.

In [5]:
weak_system = (
    "Você responde perguntas de direito penal brasileiro com base no contexto fornecido. "
    "Responda em formato JSON."
)
weak_user = (
    f"Contexto:\n{results[0].citation}: {results[0].text}\n\n"
    f"{results[1].citation}: {results[1].text}\n\n"
    f"Pergunta: {QUESTION}"
)
raw_weak_json = llm.generate(weak_user, system=weak_system, format="json")
print("--- Saída bruta (JSON não-guiado) ---")
print(raw_weak_json)

--- Saída bruta (JSON não-guiado) ---
{ "resposta": "A pena para quem mata alguém varia dependendo da circunstância do crime e da intenção do agente, mas geralmente é classificada como homicídio doloso. A pena por homicídio doloso pode variar de 12 a 30 anos de reclusão, conforme o Código Penal Brasileiro." }


In [6]:
parsed_weak = parse_answer(raw_weak_json)
print(f"parse_ok:   {parsed_weak.parse_ok}")
print(f"abstained:  {parsed_weak.abstained}")
print(f"citations:  {parsed_weak.citations}")
print(f"confidence: {parsed_weak.confidence}")

parse_ok:   True
abstained:  False
citations:  []
confidence: 0.0


**Leitura:** `format="json"` garante que a saída seja um objeto JSON válido, então
`parse_answer` agora consegue extrair *algum* objeto (`parse_ok=True`). Mas sem o contrato
de chaves fixado no system prompt, o modelo é livre para nomear as chaves como quiser
(`"resposta"`, `"fonte"`, `"dispositivos"`, ...) em vez de exatamente `"answer"`/
`"citations"`/`"abstained"`. Como `parse_answer` procura as chaves exatas do contrato,
`citations` frequentemente sai vazio ou `abstained` sai com o default (`False`) mesmo sem
o modelo ter se posicionado sobre isso — **JSON válido não é o mesmo que schema correto**.
Essa é a segunda lição de iteração.

## Técnica 3 — Few-shot + saída controlada por schema

A versão final (a mesma usada em produção por `answer_question`) resolve os dois problemas
de uma vez:

- **Few-shot / *example-guided*:** `SYSTEM_PROMPT` (abaixo) embute um **exemplo preenchido**
  de pergunta-resposta em JSON — o modelo vê a forma exata esperada antes de gerar a sua.
- **Saída restrita por schema:** a chamada usa `format=<json schema>` do Ollama (não apenas
  `"json"`), que restringe estruturalmente a geração ao formato exato — chaves, tipos,
  campos obrigatórios.
- **Ids citáveis explícitos:** `build_user_prompt` lista os ids exatos disponíveis para
  citação (`"Ids disponíveis para citação: CP:art121, CP:art155"`), removendo qualquer
  ambiguidade sobre como o modelo deve referenciar cada dispositivo.

In [7]:
print(SYSTEM_PROMPT)

Você é um assistente que explica Direito Penal brasileiro a partir de trechos de normas fornecidos abaixo. Isto é uma explicação informativa, NÃO é aconselhamento jurídico.

Regras obrigatórias:
1. Responda SOMENTE com base nas PROVISÕES fornecidas no contexto. Não use conhecimento externo nem invente dispositivos.
2. Se as provisões fornecidas respondem à pergunta, RESPONDA (defina "abstained": false) e liste em "citations" as tags dos dispositivos que você usou, usando o id exato de cada provisão (por exemplo "CP:art121", sem colchetes).
3. Abstenha-se ("abstained": true) SOMENTE quando as provisões não permitirem responder; nesse caso explique brevemente por quê.
4. Responda em português do Brasil.
5. Responda ESTRITAMENTE em JSON, sem texto fora do objeto JSON, com exatamente estas chaves: "answer" (string), "citations" (lista de ids como "CP:art121"), "hierarchy_notes" (string, pode ser vazia), "abstained" (booleano), "confidence" (número entre 0 e 1).

EXEMPLO de resposta correta

In [8]:
user_prompt = build_user_prompt(QUESTION, results)
print(user_prompt)

PROVISÕES:

[CP:art121] (CP art. 121, situação: alterado)
Matar alguem:
Pena - reclusão, de seis a vinte anos.
Caso de diminuição de pena
§
1º Se o agente comete o crime impelido por motivo de relevante valor social ou moral, ou
sob o domínio de violenta emoção, logo em seguida a injusta provocação da vítima, o
juiz pode reduzir a pena de um sexto a um terço.
Homicídio qualificado
§
2° Se o homicídio é cometido:
I -
mediante paga ou promessa de recompensa, ou por outro motivo torpe;
II
- por motivo futil;
III
- com emprego de veneno, fogo, explosivo, asfixia, tortura ou outro meio insidioso ou
cruel, ou de que possa resultar perigo comum;
IV
- à traição, de emboscada, ou mediante dissimulação ou outro recurso que dificulte ou
torne impossivel a defesa do ofendido;
V - para assegurar a execução, a ocultação, a impunidade ou vantagem de outro
crime:
Pena - reclusão, de doze a
trinta anos.
Feminicídio
(Incluído pela Lei nº
13.104, de 2015)
VI -
(Revogado pela Lei nº
14.994, de 2024)
VII 

In [9]:
# O mesmo JSON schema usado internamente por direito_dados.generation.rag.answer_question
# (reproduzido aqui para tornar visível o contrato de saída que restringe a geração).
ANSWER_SCHEMA = {
    "type": "object",
    "properties": {
        "answer": {"type": "string"},
        "citations": {"type": "array", "items": {"type": "string"}},
        "hierarchy_notes": {"type": "string"},
        "abstained": {"type": "boolean"},
        "confidence": {"type": "number"},
    },
    "required": ["answer", "citations", "abstained", "confidence"],
}

for attempt in range(1, 4):
    raw_schema = llm.generate(user_prompt, system=SYSTEM_PROMPT, format=ANSWER_SCHEMA)
    parsed_schema = parse_answer(raw_schema)
    print(f"--- tentativa {attempt} ---")
    print(raw_schema)
    if not parsed_schema.abstained and "CP:art121" in parsed_schema.citations:
        break

--- tentativa 1 ---
{ "answer": "A pena para quem mata alguém varia dependendo do contexto e da intenção, mas em geral é considerado homicídio e pode ter penas de prisão variando de alguns anos a até mesmo pena de morte em alguns países.", "citations": ["CP:art121", "CP:art155"], "abstained": false, "confidence": 1 }


In [10]:
print(f"parse_ok:   {parsed_schema.parse_ok}")
print(f"abstained:  {parsed_schema.abstained}")
print(f"citations:  {parsed_schema.citations}")
print(f"confidence: {parsed_schema.confidence}")
print(f"answer:     {parsed_schema.answer}")

parse_ok:   True
abstained:  False
citations:  ['CP:art121', 'CP:art155']
confidence: 1.0
answer:     A pena para quem mata alguém varia dependendo do contexto e da intenção, mas em geral é considerado homicídio e pode ter penas de prisão variando de alguns anos a até mesmo pena de morte em alguns países.


**Leitura:** com o schema explícito, `citations` normalmente vem preenchido com o id exato
(`"CP:art121"`) e `abstained` sai `False`, coerente com o contexto ter respondido à
pergunta. Esse é o critério de qualidade que motivou toda a iteração: **um schema explícito
+ exemplo preenchido + listagem de ids citáveis é o que faz `citations` sair correto de
forma confiável — mas "confiável" não é "determinístico".**

Note a tentativa acima: mesmo com o schema fixo, `llama3.1:8b` (um modelo local de 8B
parâmetros, quantizado) às vezes erra o id citado (confunde `CP:art121` com um dispositivo
vizinho) ou arredonda a pena incorretamente, embora a *estrutura* JSON esteja sempre
correta. Isso é um limite honesto do modelo local, não do prompt — o schema resolve o
problema estrutural (parseabilidade/forma), não o problema de raciocínio/precisão factual
do modelo. É exatamente o tipo de observação que motiva comparar com um baseline em nuvem
(ver `c05_rag_pipeline.ipynb`, seção de segurança/limitações).

### Abstenção coerente: mesma técnica, contexto insuficiente

O contrato de saída não deveria fazer o modelo "inventar" `abstained=false` quando não há
base. Testamos a mesma técnica 3 (few-shot + schema) mas com contexto vazio —
`build_user_prompt` gera, nesse caso, uma instrução explícita para abster-se.

In [11]:
empty_prompt = build_user_prompt("Qual o prazo prescricional do IPTU?", [])
print(empty_prompt)

PROVISÕES: nenhuma provisão relevante foi encontrada na base de normas para esta pergunta. Não há contexto para responder.
Responda com "abstained": true, explicando que não há base normativa recuperada.

PERGUNTA: Qual o prazo prescricional do IPTU?


In [12]:
raw_abstain = llm.generate(empty_prompt, system=SYSTEM_PROMPT, format=ANSWER_SCHEMA)
print("--- Saída bruta (schema, contexto vazio) ---")
print(raw_abstain)

parsed_abstain = parse_answer(raw_abstain)
print(f"\nparse_ok:   {parsed_abstain.parse_ok}")
print(f"abstained:  {parsed_abstain.abstained}")
print(f"citations:  {parsed_abstain.citations}")

--- Saída bruta (schema, contexto vazio) ---
{"answer": "", "citations": [], "abstained": true, "confidence": 1.0}

parse_ok:   True
abstained:  True
citations:  []


**Leitura:** sem provisões no contexto, o modelo define `abstained=true` e não inventa
`citations` — o critério de qualidade (citações corretas *e* abstenção coerente) é
satisfeito nos dois sentidos: cita quando pode, abstém-se quando não pode.

## Resumo da iteração

| Técnica | Mecanismo | `parse_ok` | `citations` preenchido corretamente? | `abstained` coerente? |
|---|---|---|---|---|
| 1. Zero-shot | instrução livre, sem `format` | não | não (nem chega a parsear) | não |
| 2. JSON não-guiado | `format="json"`, sem contrato de chaves | sim | não confiável (chaves divergentes) | não confiável |
| 3. Few-shot + schema | `SYSTEM_PROMPT` (exemplo) + `format=<schema>` + ids explícitos | sim | sim | sim |

Esse é exatamente o caminho que `direito_dados.generation.rag.answer_question` percorre em
produção: constrói o prompt com `build_user_prompt` (que já lista os ids citáveis), chama o
modelo com `system=SYSTEM_PROMPT` e `format=<schema idêntico ao ANSWER_SCHEMA acima>`, e
passa a saída por `parse_answer` — cujo fail-safe para abstenção (visto na Técnica 1) é a
última linha de defesa caso, mesmo com schema, o modelo produza algo não-parseável.

## Conclusão

Esta notebook demonstrou, com chamadas reais ao Ollama (`llama3.1:8b`), três técnicas de
prompting — zero-shot, JSON não-guiado e few-shot com saída restrita por schema — e como
cada uma se comporta contra um critério de qualidade objetivo e verificável
programaticamente: `citations` preenchido com os ids corretos e `abstained` coerente com o
contexto. A técnica final, usada em produção pelo pipeline RAG (`c05_rag_pipeline.ipynb`),
é a única que passa de forma confiável — e mesmo ela se apoia em `parse_answer` como rede de
segurança para saída malformada, nunca fabricando uma resposta estruturada a partir de lixo.